In [ ]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers datasets

In [ ]:
print(next(model.parameters()).device)

In [ ]:
prompt = "Explain Retrieval-Augmented Generation in simple words."

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=100
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)

In [ ]:
!pip install -q datasets

In [ ]:
from datasets import load_dataset
import pandas as pd

In [ ]:
dataset = load_dataset("deepmind/narrativeqa")

In [ ]:
print(dataset)

In [ ]:
test_dataset = dataset["test"]

print("Total Test Samples:", len(test_dataset))

In [ ]:
subset_dataset = test_dataset.select(range(200))

print("Subset Size:", len(subset_dataset))

In [ ]:
sample = subset_dataset[0]

print(sample)

In [ ]:
document_text = sample["document"]["text"]
question_text = sample["question"]["text"]
reference_answer = sample["answers"][0]["text"]

print("QUESTION:\n")
print(question_text)

print("\nREFERENCE ANSWER:\n")
print(reference_answer)

print("\nDOCUMENT LENGTH:\n")
print(len(document_text))

In [ ]:
processed_data = []

for item in subset_dataset:
    processed_item = {
        "document": item["document"]["text"],
        "question": item["question"]["text"],
        "reference_answer": item["answers"][0]["text"]
    }

    processed_data.append(processed_item)

print("Processed Samples:", len(processed_data))

In [ ]:
processed_data[0]

In [ ]:
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embedding Model Loaded")

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50
)

print("✅ Text Splitter Ready")

In [ ]:
processed_data = []

for item in subset_dataset:
    processed_item = {
        "document": item["document"]["text"],
        "question": item["question"]["text"],
        "reference_answer": item["answers"][0]["text"]
    }

    processed_data.append(processed_item)

print("✅ Processed Samples:", len(processed_data))

In [ ]:
sample_document = processed_data[0]["document"]

chunks = text_splitter.split_text(sample_document)

print("Total Chunks:", len(chunks))

print("\nFIRST CHUNK:\n")
print(chunks[0][:500])

In [ ]:
vector_store = FAISS.from_texts(
    texts=chunks,
    embedding=embedding_model
)

print("✅ FAISS Vector Store Created")

In [ ]:
question = processed_data[0]["question"]

retrieved_docs = vector_store.similarity_search(
    question,
    k=5
)

print("Retrieved Chunks:", len(retrieved_docs))

print("\nFIRST RETRIEVED CHUNK:\n")
print(retrieved_docs[0].page_content[:500])

In [ ]:
def create_rag_prompt(context, question):
    prompt = f"""
You are an intelligent AI assistant.

Use ONLY the provided context to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""
    return prompt

print("✅ Prompt Template Ready")

In [ ]:
retrieved_contexts = [doc.page_content for doc in retrieved_docs]

combined_context = "\n\n".join(retrieved_contexts)

print(combined_context[:1000])

In [ ]:
rag_prompt = create_rag_prompt(
    combined_context,
    question
)

print(rag_prompt[:1500])

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
!pip install -U bitsandbytes

In [ ]:
sample_document = processed_data[0]["document"]

chunks = text_splitter.split_text(sample_document)

print("Total Chunks:", len(chunks))

print("\nFIRST CHUNK:\n")
print(chunks[0][:500])

In [ ]:
vector_store = FAISS.from_texts(
    texts=chunks,
    embedding=embedding_model
)

print("✅ FAISS Vector Store Created")

In [ ]:
question = processed_data[0]["question"]

retrieved_docs = vector_store.similarity_search(
    question,
    k=5
)

print("Retrieved Chunks:", len(retrieved_docs))

print("\nFIRST RETRIEVED CHUNK:\n")
print(retrieved_docs[0].page_content[:500])

In [ ]:
def create_rag_prompt(context, question):
    prompt = f"""
You are an intelligent AI assistant.

Use ONLY the provided context to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""
    return prompt

print("✅ Prompt Template Ready")

In [ ]:
retrieved_contexts = [doc.page_content for doc in retrieved_docs]

combined_context = "\n\n".join(retrieved_contexts)

print(combined_context[:1000])

In [ ]:
rag_prompt = create_rag_prompt(
    combined_context,
    question
)

print(rag_prompt[:1500])

In [ ]:
inputs = tokenizer(
    rag_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=2048
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    temperature=0.3
)

generated_answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(generated_answer)

In [ ]:
import json
from tqdm import tqdm

In [ ]:
def clean_generated_answer(full_output):
    if "Answer:" in full_output:
        return full_output.split("Answer:")[-1].strip()
    return full_output.strip()

print("✅ Answer Cleaning Function Ready")

In [ ]:
test_rag_results = []

small_subset = processed_data[:3]

for item in tqdm(small_subset):

    document = item["document"]
    question = item["question"]
    reference_answer = item["reference_answer"]

    # Split document into chunks
    chunks = text_splitter.split_text(document)

    # Create FAISS vector store
    vector_store = FAISS.from_texts(
        texts=chunks,
        embedding=embedding_model
    )

    # Retrieve top-5 chunks
    retrieved_docs = vector_store.similarity_search(
        question,
        k=5
    )

    # Extract retrieved contexts
    retrieved_contexts = [
        doc.page_content for doc in retrieved_docs
    ]

    # Combine contexts
    combined_context = "\n\n".join(retrieved_contexts)

    # Create RAG prompt
    rag_prompt = create_rag_prompt(
        combined_context,
        question
    )

    # Tokenize input
    inputs = tokenizer(
        rag_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to("cuda")

    # Generate answer
    outputs = model.generate(
        **inputs,
        max_new_tokens=150
    )

    # Decode output
    generated_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Clean answer
    cleaned_answer = clean_generated_answer(
        generated_text
    )

    # Store result
    result = {
        "question": question,
        "generated_answer": cleaned_answer,
        "reference_answer": reference_answer,
        "retrieved_contexts": retrieved_contexts
    }

    test_rag_results.append(result)

print("✅ Small RAG Test Completed")

In [ ]:
test_rag_results[0]

In [ ]:
import os

os.makedirs("results", exist_ok=True)

print("✅ Results Folder Ready")

In [ ]:
rag_results = []

full_processed_data = processed_data[:200]

for item in tqdm(full_processed_data):

    try:

        document = item["document"]
        question = item["question"]
        reference_answer = item["reference_answer"]

        # Split document into chunks
        chunks = text_splitter.split_text(document)

        # Speed optimization
        chunks = chunks[:30]

        # Create FAISS vector store
        vector_store = FAISS.from_texts(
            texts=chunks,
            embedding=embedding_model
        )

        # Retrieve top-3 chunks
        retrieved_docs = vector_store.similarity_search(
            question,
            k=3
        )

        # Extract retrieved contexts
        retrieved_contexts = [
            doc.page_content for doc in retrieved_docs
        ]

        # Combine contexts
        combined_context = "\n\n".join(retrieved_contexts)

        # Create prompt
        rag_prompt = create_rag_prompt(
            combined_context,
            question
        )

        # Tokenize input
        inputs = tokenizer(
            rag_prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to("cuda")

        # Generate answer
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=False
        )

        # Decode generated output
        generated_text = tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        )

        # Clean generated answer
        cleaned_answer = clean_generated_answer(
            generated_text
        )

        # Store result
        result = {
            "question": question,
            "generated_answer": cleaned_answer,
            "reference_answer": reference_answer,
            "retrieved_contexts": retrieved_contexts
        }

        rag_results.append(result)

    except Exception as e:

        print(f"Error processing question: {question}")
        print(e)

print("✅ Full 200-Sample RAG Pipeline Completed")

In [ ]:
len(rag_results)

In [ ]:
with open("results/rag_answers.json", "w") as f:
    json.dump(rag_results, f, indent=4)

print("✅ rag_answers.json Saved Successfully")

In [ ]:
with open("results/rag_answers.json", "r") as f:
    loaded_rag_results = json.load(f)

print("Total Saved Results:", len(loaded_rag_results))

In [ ]:
def create_long_context_prompt(document, question):

    prompt = f"""
You are an intelligent AI assistant.

Use ONLY the provided document to answer the question.

Document:
{document}

Question:
{question}

Answer:
"""

    return prompt

print("✅ Long Context Prompt Function Ready")

In [ ]:
sample_document = processed_data[0]["document"]

tokens = tokenizer.encode(sample_document)

print("Original Token Length:", len(tokens))

truncated_tokens = tokens[:6000]

print("Truncated Token Length:", len(truncated_tokens))

In [ ]:
truncated_document = tokenizer.decode(
    truncated_tokens,
    skip_special_tokens=True
)

print(truncated_document[:1000])

In [ ]:
question = processed_data[0]["question"]

long_prompt = create_long_context_prompt(
    truncated_document,
    question
)

print(long_prompt[:1500])

In [ ]:
inputs = tokenizer(
    long_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=2048
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=False
)

generated_text = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

cleaned_answer = clean_generated_answer(
    generated_text
)

print(cleaned_answer)

In [ ]:
long_context_results = []

full_processed_data = processed_data[:200]

for item in tqdm(full_processed_data):

    try:

        document = item["document"]
        question = item["question"]
        reference_answer = item["reference_answer"]

        # Tokenize document
        document_tokens = tokenizer.encode(document)

        # Truncate for safety
        truncated_tokens = document_tokens[:6000]

        # Convert back to text
        truncated_document = tokenizer.decode(
            truncated_tokens,
            skip_special_tokens=True
        )

        # Create long-context prompt
        long_prompt = create_long_context_prompt(
            truncated_document,
            question
        )

        # Tokenize input
        inputs = tokenizer(
            long_prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to("cuda")

        # Generate answer
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=False
        )

        # Decode output
        generated_text = tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        )

        # Clean answer
        cleaned_answer = clean_generated_answer(
            generated_text
        )

        # Store result
        result = {
            "question": question,
            "generated_answer": cleaned_answer,
            "reference_answer": reference_answer
        }

        long_context_results.append(result)

    except Exception as e:

        print(f"Error processing question: {question}")
        print(e)

print("✅ Full Long-Context Pipeline Completed")

In [ ]:
len(long_context_results)

In [ ]:
with open("results/long_context_answers.json", "w") as f:
    json.dump(long_context_results, f, indent=4)

print("✅ long_context_answers.json Saved Successfully")

In [ ]:
with open("results/long_context_answers.json", "r") as f:
    loaded_long_results = json.load(f)

print("Total Saved Results:", len(loaded_long_results))

In [ ]:
!pip install -q rouge-score bert-score ragas

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score
import numpy as np

In [ ]:
test_rag_subset = rag_results[:10]
test_long_subset = long_context_results[:10]

print("RAG Test Samples:", len(test_rag_subset))
print("Long Context Test Samples:", len(test_long_subset))

In [ ]:
rouge = rouge_scorer.RougeScorer(
    ['rouge1', 'rougeL'],
    use_stemmer=True
)

print("✅ ROUGE Scorer Ready")

In [ ]:
sample_prediction = test_rag_subset[0]["generated_answer"]
sample_reference = test_rag_subset[0]["reference_answer"]

scores = rouge.score(
    sample_reference,
    sample_prediction
)

print(scores)

In [ ]:
predictions = [
    test_rag_subset[0]["generated_answer"]
]

references = [
    test_rag_subset[0]["reference_answer"]
]

P, R, F1 = score(
    predictions,
    references,
    lang="en",
    verbose=True
)

print("BERTScore F1:", F1.mean().item())

In [ ]:
rag_rouge_scores = []

for item in tqdm(rag_results):

    prediction = item["generated_answer"]
    reference = item["reference_answer"]

    scores = rouge.score(
        reference,
        prediction
    )

    result = {
        "rouge1_fmeasure": scores["rouge1"].fmeasure,
        "rougeL_fmeasure": scores["rougeL"].fmeasure
    }

    rag_rouge_scores.append(result)

print("✅ RAG ROUGE Evaluation Completed")

In [ ]:
long_rouge_scores = []

for item in tqdm(long_context_results):

    prediction = item["generated_answer"]
    reference = item["reference_answer"]

    scores = rouge.score(
        reference,
        prediction
    )

    result = {
        "rouge1_fmeasure": scores["rouge1"].fmeasure,
        "rougeL_fmeasure": scores["rougeL"].fmeasure
    }

    long_rouge_scores.append(result)

print("✅ Long-Context ROUGE Evaluation Completed")

In [ ]:
rag_avg_rouge1 = np.mean([
    x["rouge1_fmeasure"]
    for x in rag_rouge_scores
])

rag_avg_rougeL = np.mean([
    x["rougeL_fmeasure"]
    for x in rag_rouge_scores
])

long_avg_rouge1 = np.mean([
    x["rouge1_fmeasure"]
    for x in long_rouge_scores
])

long_avg_rougeL = np.mean([
    x["rougeL_fmeasure"]
    for x in long_rouge_scores
])

print("RAG ROUGE-1:", rag_avg_rouge1)
print("RAG ROUGE-L:", rag_avg_rougeL)

print("Long Context ROUGE-1:", long_avg_rouge1)
print("Long Context ROUGE-L:", long_avg_rougeL)

In [ ]:
evaluation_scores = {
    "rag": {
        "average_rouge1_fmeasure": float(rag_avg_rouge1),
        "average_rougeL_fmeasure": float(rag_avg_rougeL)
    },
    "long_context": {
        "average_rouge1_fmeasure": float(long_avg_rouge1),
        "average_rougeL_fmeasure": float(long_avg_rougeL)
    }
}

print(evaluation_scores)

In [ ]:
with open("results/evaluation_scores.json", "w") as f:
    json.dump(evaluation_scores, f, indent=4)

print("✅ evaluation_scores.json Saved Successfully")

In [ ]:
with open("results/evaluation_scores.json", "r") as f:
    loaded_scores = json.load(f)

print(loaded_scores)

In [ ]:
position_results = {
    "beginning": [],
    "middle": [],
    "end": []
}

print("✅ Position Analysis Structure Ready")

In [ ]:
def get_document_sections(document):

    tokens = tokenizer.encode(document)

    total_tokens = len(tokens)

    beginning = tokenizer.decode(
        tokens[:2000],
        skip_special_tokens=True
    )

    middle_start = total_tokens // 2

    middle = tokenizer.decode(
        tokens[middle_start:middle_start + 2000],
        skip_special_tokens=True
    )

    end = tokenizer.decode(
        tokens[-2000:],
        skip_special_tokens=True
    )

    return beginning, middle, end

print("✅ Position Extraction Function Ready")

In [ ]:
sample_document = processed_data[0]["document"]

beginning, middle, end = get_document_sections(
    sample_document
)

print("Beginning Length:", len(beginning))
print("Middle Length:", len(middle))
print("End Length:", len(end))

In [ ]:
def create_position_prompt(context, question):

    prompt = f"""
You are an intelligent AI assistant.

Use ONLY the provided context to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

    return prompt

print("✅ Position Prompt Function Ready")

In [ ]:
question = processed_data[0]["question"]

prompt = create_position_prompt(
    middle,
    question
)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=1024
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=False
)

generated_text = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

cleaned_answer = clean_generated_answer(
    generated_text
)

print(cleaned_answer)

In [ ]:
position_results = {
    "beginning": [],
    "middle": [],
    "end": []
}

analysis_subset = processed_data[:50]

for item in tqdm(analysis_subset):

    try:

        document = item["document"]
        question = item["question"]
        reference_answer = item["reference_answer"]

        # Extract sections
        beginning, middle, end = get_document_sections(
            document
        )

        sections = {
            "beginning": beginning,
            "middle": middle,
            "end": end
        }

        for position_name, context in sections.items():

            # Create prompt
            prompt = create_position_prompt(
                context,
                question
            )

            # Tokenize
            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=1024
            ).to("cuda")

            # Generate
            outputs = model.generate(
                **inputs,
                max_new_tokens=60,
                do_sample=False
            )

            # Decode
            generated_text = tokenizer.decode(
                outputs[0],
                skip_special_tokens=True
            )

            # Clean answer
            cleaned_answer = clean_generated_answer(
                generated_text
            )

            # ROUGE evaluation
            scores = rouge.score(
                reference_answer,
                cleaned_answer
            )

            rouge_score_value = scores[
                "rouge1"
            ].fmeasure

            # Store score
            position_results[
                position_name
            ].append(
                rouge_score_value
            )

    except Exception as e:

        print("Error:", e)

print("✅ Position Sensitivity Analysis Completed")

In [ ]:
print("Beginning Scores:", len(position_results["beginning"]))
print("Middle Scores:", len(position_results["middle"]))
print("End Scores:", len(position_results["end"]))

In [ ]:
beginning_avg = np.mean(
    position_results["beginning"]
)

middle_avg = np.mean(
    position_results["middle"]
)

end_avg = np.mean(
    position_results["end"]
)

print("Beginning Average:", beginning_avg)
print("Middle Average:", middle_avg)
print("End Average:", end_avg)

In [ ]:
position_analysis = {
    "beginning_average_rouge1": float(beginning_avg),
    "middle_average_rouge1": float(middle_avg),
    "end_average_rouge1": float(end_avg)
}

print(position_analysis)

In [ ]:
with open(
    "results/position_sensitivity_analysis.json",
    "w"
) as f:

    json.dump(
        position_analysis,
        f,
        indent=4
    )

print(
    "✅ position_sensitivity_analysis.json Saved"
)

In [ ]:
import matplotlib.pyplot as plt

positions = [
    "Beginning",
    "Middle",
    "End"
]

scores = [
    beginning_avg,
    middle_avg,
    end_avg
]

plt.figure(figsize=(8, 5))

plt.bar(
    positions,
    scores
)

plt.xlabel("Document Position")
plt.ylabel("Average ROUGE-1 Score")
plt.title("Position Sensitivity Analysis")

plt.savefig(
    "results/position_sensitivity_chart.png"
)

plt.show()

print("✅ Position Sensitivity Chart Saved")

In [ ]:
import os

print(
    os.listdir("results")
)

In [ ]:
import time
import torch
import psutil

In [ ]:
from datasets import load_dataset

dataset = load_dataset("deepmind/narrativeqa")

test_dataset = dataset["test"]

subset_dataset = test_dataset.select(range(200))

In [ ]:
processed_data = []

for item in subset_dataset:

    processed_item = {
        "document": item["document"]["text"],
        "question": item["question"]["text"],
        "reference_answer": item["answers"][0]["text"]
    }

    processed_data.append(processed_item)

print(
    "✅ Processed Samples:",
    len(processed_data)
)

In [ ]:
benchmark_sample = processed_data[0]

print(
    benchmark_sample["question"]
)

In [ ]:
!pip install -q langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50
)

print("✅ text_splitter Restored")

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embedding Model Restored")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("✅ tokenizer and model Restored")

In [ ]:
import time
import torch
import psutil

sample_prompt = """
You are an intelligent AI assistant.

Answer the following question.

Question:
Who is Mark Hunter?

Answer:
"""

start_time = time.time()

inputs = tokenizer(
    sample_prompt,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=False
)

end_time = time.time()

latency = end_time - start_time

generated_tokens = outputs[0].shape[0]

throughput = generated_tokens / latency

gpu_memory = torch.cuda.memory_allocated()

gpu_memory_gb = gpu_memory / (1024 ** 3)

cpu_memory = psutil.virtual_memory().used

cpu_memory_gb = cpu_memory / (1024 ** 3)

performance_metrics = {
    "latency_seconds": float(latency),
    "throughput_tokens_per_second": float(throughput),
    "gpu_memory_gb": float(gpu_memory_gb),
    "cpu_memory_gb": float(cpu_memory_gb)
}

print(performance_metrics)

In [ ]:
import os

os.makedirs(
    "results",
    exist_ok=True
)

print("✅ results folder recreated")

In [ ]:
import json
with open(
    "results/performance_metrics.json",
    "w"
) as f:

    json.dump(
        performance_metrics,
        f,
        indent=4
    )

print(
    "✅ performance_metrics.json Saved"
)

In [ ]:
import os

print(
    os.listdir("results")
)

In [ ]:
from google.colab import files

files.download("results/performance_metrics.json")

In [ ]:
import os

os.makedirs("results", exist_ok=True)

print("✅ results folder ready")

In [ ]:
import os

os.makedirs(
    "results",
    exist_ok=True
)

print("✅ results folder created")

In [ ]:
rag_results = []

full_processed_data = processed_data[:200]

for item in tqdm(full_processed_data):

    try:

        document = item["document"]
        question = item["question"]
        reference_answer = item["reference_answer"]

        chunks = text_splitter.split_text(document)

        chunks = chunks[:30]

        vector_store = FAISS.from_texts(
            texts=chunks,
            embedding=embedding_model
        )

        retrieved_docs = vector_store.similarity_search(
            question,
            k=3
        )

        retrieved_contexts = [
            doc.page_content for doc in retrieved_docs
        ]

        combined_context = "\n\n".join(
            retrieved_contexts
        )

        rag_prompt = create_rag_prompt(
            combined_context,
            question
        )

        inputs = tokenizer(
            rag_prompt,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to("cuda")

        outputs = model.generate(
            **inputs,
            max_new_tokens=60,
            do_sample=False
        )

        generated_text = tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        )

        cleaned_answer = clean_generated_answer(
            generated_text
        )

        result = {
            "question": question,
            "generated_answer": cleaned_answer,
            "reference_answer": reference_answer,
            "retrieved_contexts": retrieved_contexts
        }

        rag_results.append(result)

    except Exception as e:

        print(e)

print("✅ rag_results Ready")

In [ ]:
from datasets import load_dataset

dataset = load_dataset("deepmind/narrativeqa")

test_dataset = dataset["test"]

subset_dataset = test_dataset.select(range(200))

print("✅ Dataset Loaded")

In [ ]:
processed_data = []

for item in subset_dataset:

    processed_item = {
        "document": item["document"]["text"],
        "question": item["question"]["text"],
        "reference_answer": item["answers"][0]["text"]
    }

    processed_data.append(processed_item)

print("✅ processed_data Ready")
print("Total Samples:", len(processed_data))

In [ ]:
!pip install -q langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50
)

print("✅ text_splitter Ready")

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ embedding_model Ready")

In [ ]:
!pip install -q langchain-community faiss-cpu

In [ ]:
from langchain_community.vectorstores import FAISS

print("✅ FAISS Imported Successfully")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50
)

print("✅ text_splitter Ready")

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ embedding_model Ready")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("✅ model + tokenizer Ready")

In [ ]:
from datasets import load_dataset

dataset = load_dataset("deepmind/narrativeqa")

test_dataset = dataset["test"]

subset_dataset = test_dataset.select(range(200))

print("✅ Dataset Ready")

In [ ]:
processed_data = []

for item in subset_dataset:

    processed_item = {
        "document": item["document"]["text"],
        "question": item["question"]["text"],
        "reference_answer": item["answers"][0]["text"]
    }

    processed_data.append(processed_item)

print("✅ processed_data Ready")
print("Total Samples:", len(processed_data))

In [ ]:
def create_rag_prompt(context, question):

    prompt = f"""
You are an intelligent AI assistant.

Use ONLY the provided context to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""

    return prompt


def clean_generated_answer(text):

    if "Answer:" in text:
        text = text.split("Answer:")[-1]

    return text.strip()

print("✅ Functions Ready")

In [ ]:
from google.colab import files

files.download(
    "results/rag_answers.json"
)

In [ ]:
from tqdm import tqdm

In [ ]:
long_context_results = []

In [ ]:
len(processed_data)

In [ ]:
print(type(model))

In [ ]:
print(type(tokenizer))

In [ ]:
print(type(text_splitter))

In [ ]:
print(type(embedding_model))

In [ ]:
!pip install -q langchain-huggingface